# Book generation for Transfer Learning experiments -- End to end
- Generate the books and the corresponding question/answer pairs
- If the data already exists, only load books and regenerate QA pairs
- Duration for generating 100 books: 1/2 days

In [ ]:
from pathlib import Path
git_repo_filepath = str(Path.cwd().resolve().parents[2])  # repository root (this notebook: src/In_vitro_experiments/experiments)
import sys

# Add the project root to sys.path
if str(Path(git_repo_filepath) / 'src') not in sys.path:
    sys.path.insert(0, str(Path(git_repo_filepath) / 'src')) # Insert at the beginning to prioritize it

print(f"Project root added to sys.path: {git_repo_filepath}")
print(f"Current sys.path: {sys.path}")
%load_ext autoreload
%autoreload 2
import os

# Set CUDA_VISIBLE_DEVICES to restrict GPU visibility to devices 0 and 1
# This MUST be done BEFORE importing torch, transformers, or any other CUDA-dependent library.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
from In_vitro_experiments.generation.benchmark_generation_wrapper import BenchmarkGenerationWrapper

book_parameters = {'indexing': 'default', 'nb_summaries': 0}
#data_folder = Path(git_repo_filepath) / 'data' / 'narratives_transfer'
data_folder = Path(git_repo_filepath) / 'data' / 'narratives_transfer'
env_file = Path(git_repo_filepath) / '.env'
model = 'claude-sonnet-4-6' 

print("Generation with Claude -- 10 events")
#prompt_parameters = {'nb_events': 10, 'name_universe': 'default', 'name_styles': 'concise', 'seed': 0, 'distribution_events': {'name': 'geometric', 'param': 0.3}}
model_parameters = {'model_name': model, 'max_new_tokens': 4096, 'itermax': 10} # itermax is integer, 1 for a single try
prompt_parameters = {
    'nb_events': 10,
    'name_universe': 'default',
    'name_styles': 'concise',
    'seed': 200,
    'disjoint_book_id': 10,
    'n_total_books': 20,  
    'distribution_events': {'name': 'geometric', 'param': 0.3}
}

benchmark_claude_10_def = BenchmarkGenerationWrapper(prompt_parameters, model_parameters, book_parameters, data_folder, env_file,rechecking=True)
benchmark_claude_10_def.df_book_groundtruth[['chapter', 'date', 'location', 'entity', 'content']]

In [ ]:
# for name, benchmark in benchmarks.items():
df = benchmark_claude_10_def.df_book_groundtruth[['chapter', 'date', 'location', 'entity', 'content']]
for col in df.columns:
    uniques = df[col].dropna().unique()
    print(f"{col}: {len(uniques)} distinct")
    print(f"  values: {list(uniques)}\n")

display(df)

# Build 6 disj books:

In [ ]:
from pathlib import Path
git_repo_filepath = str(Path.cwd().resolve().parents[2])  # repository root (this notebook: src/In_vitro_experiments/experiments)
import sys

# Add the project root to sys.path
if str(Path(git_repo_filepath) / 'src') not in sys.path:
    sys.path.insert(0, str(Path(git_repo_filepath) / 'src')) # Insert at the beginning to prioritize it

print(f"Project root added to sys.path: {git_repo_filepath}")
print(f"Current sys.path: {sys.path}")
%load_ext autoreload
%autoreload 2
import os

# Set CUDA_VISIBLE_DEVICES to restrict GPU visibility to devices 0 and 1
# This MUST be done BEFORE importing torch, transformers, or any other CUDA-dependent library.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
from In_vitro_experiments.generation.benchmark_generation_wrapper import BenchmarkGenerationWrapper
book_parameters = {'indexing': 'default', 'nb_summaries': 0}
#data_folder = Path(git_repo_filepath) / 'data' / 'narratives_transfer'
data_folder = Path(git_repo_filepath) / 'data' / 'narratives_transfer' 
env_file = Path(git_repo_filepath) / '.env'
model = 'claude-sonnet-4-6' 

#prompt_parameters = {'nb_events': 10, 'name_universe': 'default', 'name_styles': 'concise', 'seed': 0, 'distribution_events': {'name': 'geometric', 'param': 0.3}}
model_parameters = {'model_name': model, 'max_new_tokens': 4096, 'itermax': 10} # itermax is integer, 1 for a single try
#benchmark_claude_10_def = BenchmarkGenerationWrapper(prompt_parameters, model_parameters, book_parameters, data_folder, env_file,rechecking=False)

benchmarks = {}

configs = [
    # (book_id, universe, seed)
    (1, 'default', 1),
    (2, 'default', 22),
    (3, 'scifi', 33),
    (4, 'scifi', 44),
    (5, 'news', 55),
    (6, 'news', 66),
    (7, 'default', 77),
    (8, 'default', 88),
    (9, 'default', 90),
    (10, 'default', 100),
]

for book_id, universe, seed in configs:
    prompt_parameters = {
        'nb_events': 10,
        'name_universe': universe,
        'name_styles': 'concise',
        'seed': seed,
        'disjoint_book_id': book_id,
        'n_total_books': 10,  # partition into 10, even though generating only 6
        'distribution_events': {'name': 'geometric', 'param': 0.3}
    }
    benchmarks[f'{universe}_{book_id}'] = BenchmarkGenerationWrapper(
        prompt_parameters, model_parameters, book_parameters, data_folder, env_file, rechecking=False
    )
    print(f"Book {book_id} ({universe}) done!")

In [ ]:
for name, benchmark in benchmarks.items():
    print(f"\n=== {name} ===")
    display(benchmark.df_book_groundtruth[['chapter', 'date', 'location', 'entity', 'content']])

In [ ]:
benchmark.finetuning_questions['n_items_correct_answer'].value_counts()

In [ ]:
benchmark.finetuning_questions_one_chapter['n_items_correct_answer'].value_counts()

In [ ]:
for name, benchmark in benchmarks.items():
    print(f"\n=== {name} ===")
    display(benchmark.df_book_groundtruth[['chapter', 'date', 'location', 'entity', 'content']])

# Create new percentage

In [ ]:
import json
import os
from random import shuffle

# Root output
output_root = ""

# ---- FULL CONFIG WITH SEEDS (10-event books) ----
book_configs = {
    'default_1':  {'book_id': 1,  'universe': 'default', 'seed': 1},
    'default_2':  {'book_id': 2,  'universe': 'default', 'seed': 22},
    'scifi_3':    {'book_id': 3,  'universe': 'scifi',   'seed': 33},
    'scifi_4':    {'book_id': 4,  'universe': 'scifi',   'seed': 44},
    'news_5':     {'book_id': 5,  'universe': 'news',    'seed': 55},
    'news_6':     {'book_id': 6,  'universe': 'news',    'seed': 66},
    'default_7':  {'book_id': 7,  'universe': 'default', 'seed': 77},
    'default_8':  {'book_id': 8,  'universe': 'default', 'seed': 88},
    'default_9':  {'book_id': 9,  'universe': 'default', 'seed': 90},
    'default_10': {'book_id': 10, 'universe': 'default', 'seed': 100},
}

# ---- FIXED FOLDS (one held-out per universe) ----
folds = [
    {'held_out': 'default_9', 'name': 'fold1_default'},
    {'held_out': 'scifi_4',   'name': 'fold2_scifi'},
    {'held_out': 'news_5',    'name': 'fold3_news'},
]


def build_dataset(selected_keys, held_out_key, setting_name, fold_name):
    all_jsonl = []
    held_out_seed = book_configs[held_out_key]['seed']

    print(f"\n===== {setting_name} | {fold_name} =====")
    print(f"Held-out: {held_out_key} (seed={held_out_seed})\n")

    for key in selected_keys:
        benchmark = benchmarks[key]
        if key == held_out_key:
            jsonl = benchmark.build_fine_tuning_jsonl(
                answer_in_one_chapter_only=True
            )
            print(f"{key} (SEQ ONLY) -> {len(jsonl)}")
        else:
            jsonl = benchmark.build_fine_tuning_jsonl(
                answer_in_one_chapter_only=False
            )
            print(f"{key} (SEQ+MEQ) -> {len(jsonl)}")
        all_jsonl.extend(jsonl)

    shuffle(all_jsonl)

    output_dir = f"{output_root}/{setting_name}"
    os.makedirs(output_dir, exist_ok=True)

    # 🔥 include BOTH key and seed in filename
    filepath = (
        f"{output_dir}/"
        f"{setting_name}_{fold_name}_heldout_{held_out_key}_seed_{held_out_seed}.jsonl"
    )
    with open(filepath, "w") as f:
        for item in all_jsonl:
            f.write(json.dumps(item) + "\n")

    print(f"\nTotal: {len(all_jsonl)} -> {filepath}\n")


# =========================
# 🔹 TRAINING SETS 4
# =========================
for fold in folds:
    held_out = fold['held_out']
    fold_name = fold['name']
    selected = [
        'default_2',
        'scifi_3',
        'news_6',
        held_out,
    ]
    build_dataset(selected, held_out, "training_sets_4", fold_name)

# =========================
# 🔹 TRAINING SETS 7
# =========================
for fold in folds:
    held_out = fold['held_out']
    fold_name = fold['name']
    selected = [
        'default_1', 'default_2', 'default_7', 'default_8',
        'scifi_3',
        'news_6',
        held_out,
    ]
    build_dataset(selected, held_out, "training_sets_7", fold_name)

# =========================
# 🔹 TRAINING SETS 10
# =========================
all_books = list(book_configs.keys())
for fold in folds:
    held_out = fold['held_out']
    fold_name = fold['name']
    selected = list(all_books)
    build_dataset(selected, held_out, "training_sets_10", fold_name)

# new percent with 20 books

In [ ]:
import json
import os
from random import shuffle

# book_configs must match exactly what Cell 1 loaded
book_configs = {
    'default_1':  {'book_id': 1,  'universe': 'default', 'seed': 200},
    'default_2':  {'book_id': 2,  'universe': 'default', 'seed': 201},
    'default_3':  {'book_id': 3,  'universe': 'default', 'seed': 202},
    'default_4':  {'book_id': 4,  'universe': 'default', 'seed': 203},
    'default_5':  {'book_id': 5,  'universe': 'default', 'seed': 204},
    'default_6':  {'book_id': 6,  'universe': 'default', 'seed': 205},
    'default_7':  {'book_id': 7,  'universe': 'default', 'seed': 206},
    'default_8':  {'book_id': 8,  'universe': 'default', 'seed': 207},
    'scifi_9':    {'book_id': 9,  'universe': 'scifi',   'seed': 208},
    'scifi_10':   {'book_id': 10, 'universe': 'scifi',   'seed': 209},
    'scifi_11':   {'book_id': 11, 'universe': 'scifi',   'seed': 210},
    'scifi_12':   {'book_id': 12, 'universe': 'scifi',   'seed': 211},
    'scifi_13':   {'book_id': 13, 'universe': 'scifi',   'seed': 212},
    'scifi_14':   {'book_id': 14, 'universe': 'scifi',   'seed': 213},
    'news_15':    {'book_id': 15, 'universe': 'news',    'seed': 215},
    'news_16':    {'book_id': 16, 'universe': 'news',    'seed': 216},
    'news_17':    {'book_id': 17, 'universe': 'news',    'seed': 217},
    'news_18':    {'book_id': 18, 'universe': 'news',    'seed': 218},
    'news_19':    {'book_id': 19, 'universe': 'news',    'seed': 219},
    'news_20':    {'book_id': 20, 'universe': 'news',    'seed': 220},
}

# Sanity check: make sure all keys exist in benchmarks
missing = [k for k in book_configs if k not in benchmarks]
if missing:
    print(f"WARNING: missing from benchmarks: {missing}")
    print(f"Available keys: {sorted(benchmarks.keys())}")
else:
    print(f"All {len(book_configs)} books found in benchmarks dict")

output_root = ""

folds = [
    {'held_out': 'default_5',  'name': 'fold1_default'},
    {'held_out': 'scifi_11',   'name': 'fold2_scifi'},
    {'held_out': 'news_17',    'name': 'fold3_news'},
]

def build_dataset(selected_keys, held_out_key, setting_name, fold_name):
    all_jsonl = []
    held_out_seed = book_configs[held_out_key]['seed']
    print(f"\n===== {setting_name} | {fold_name} =====")
    print(f"Held-out: {held_out_key} (seed={held_out_seed})\n")

    for key in selected_keys:
        benchmark = benchmarks[key]
        if key == held_out_key:
            jsonl = benchmark.build_fine_tuning_jsonl(answer_in_one_chapter_only=True)
            print(f"{key} (SEQ ONLY) -> {len(jsonl)}")
        else:
            jsonl = benchmark.build_fine_tuning_jsonl(answer_in_one_chapter_only=False)
            print(f"{key} (SEQ+MEQ) -> {len(jsonl)}")
        all_jsonl.extend(jsonl)

    shuffle(all_jsonl)

    output_dir = f"{output_root}/{setting_name}"
    os.makedirs(output_dir, exist_ok=True)

    filepath = (
        f"{output_dir}/"
        f"{setting_name}_{fold_name}_heldout_{held_out_key}_seed_{held_out_seed}.jsonl"
    )
    with open(filepath, "w") as f:
        for item in all_jsonl:
            f.write(json.dumps(item) + "\n")

    print(f"\nTotal: {len(all_jsonl)} -> {filepath}\n")

# =========================
# 🔹 TRAINING SETS 20
# =========================
all_books = list(book_configs.keys())
for fold in folds:
    held_out = fold['held_out']
    fold_name = fold['name']
    build_dataset(all_books, held_out, "training_sets_20", fold_name)

# MOVE TO 100 BOOKS!

In [ ]:
from pathlib import Path
git_repo_filepath = str(Path.cwd().resolve().parents[2])  # repository root (this notebook: src/In_vitro_experiments/experiments)
import sys
if str(Path(git_repo_filepath) / 'src') not in sys.path:
    sys.path.insert(0, str(Path(git_repo_filepath) / 'src'))

%load_ext autoreload
%autoreload 2

import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""

from In_vitro_experiments.generation.benchmark_generation_wrapper import BenchmarkGenerationWrapper

book_parameters = {'indexing': 'default', 'nb_summaries': 0}
data_folder = Path(git_repo_filepath) / 'data' / 'narratives_transfer'
env_file = Path(git_repo_filepath) / '.env'
model = 'claude-sonnet-4-6'
model_parameters = {'model_name': model, 'max_new_tokens': 4096, 'itermax': 10}

n_books = 100
benchmarks = {}

# === 34 default: book_ids 1–34, seeds 1000–1033 ===
for i in range(34):
    book_id = i + 1
    seed = 1000 + i
    prompt_parameters = {
        'nb_events': 10,
        'name_universe': 'default',
        'name_styles': 'concise',
        'seed': seed,
        'disjoint_book_id': book_id,
        'n_total_books': n_books,
        'distribution_events': {'name': 'geometric', 'param': 0.3}
    }
    benchmarks[f'default_{book_id}'] = BenchmarkGenerationWrapper(
        prompt_parameters, model_parameters, book_parameters, data_folder, env_file,
        rechecking=False
    )
    print(f"Book default_{book_id} (seed={seed}) done!")


# Remaining books: rechecking=True
for i in range(33):
    book_id = 35 + i
    seed = 1034 + i

    prompt_parameters = {
        'nb_events': 10,
        'name_universe': 'scifi',
        'name_styles': 'concise',
        'seed': seed,
        'disjoint_book_id': book_id,
        'n_total_books': n_books,
        'distribution_events': {'name': 'geometric', 'param': 0.3}
    }

    benchmarks[f'scifi_{book_id}'] = BenchmarkGenerationWrapper(
        prompt_parameters, model_parameters, book_parameters, data_folder, env_file,
        rechecking=False
    )

    print(f"Book scifi_{book_id} (seed={seed}) done!")

# === 33 news: book_ids 68–100, seeds 1067–1099 ===
for i in range(33):
    book_id = 68 + i
    seed = 1067 + i
    prompt_parameters = {
        'nb_events': 10,
        'name_universe': 'news',
        'name_styles': 'concise',
        'seed': seed,
        'disjoint_book_id': book_id,
        'n_total_books': n_books,
        'distribution_events': {'name': 'geometric', 'param': 0.3}
    }
    benchmarks[f'news_{book_id}'] = BenchmarkGenerationWrapper(
        prompt_parameters, model_parameters, book_parameters, data_folder, env_file,
        rechecking=False
    )
    print(f"Book news_{book_id} (seed={seed}) done!")

print(f"\nTotal benchmarks loaded: {len(benchmarks)}")
# print(f"Keys: {sorted(benchmarks.keys())}")

In [ ]:
import json
import os
from random import shuffle

# ============================================
# Build book_configs automatically for 100 books
# ============================================

book_configs = {}

# ---- default: books 1-34, seeds 1000-1033
for i in range(34):
    book_id = i + 1
    seed = 1000 + i
    key = f"default_{book_id}"

    book_configs[key] = {
        'book_id': book_id,
        'universe': 'default',
        'seed': seed
    }

# ---- scifi: books 35-67, seeds 1034-1066
for i in range(33):
    book_id = 35 + i
    seed = 1034 + i
    key = f"scifi_{book_id}"

    book_configs[key] = {
        'book_id': book_id,
        'universe': 'scifi',
        'seed': seed
    }

# ---- news: books 68-100, seeds 1067-1099
for i in range(33):
    book_id = 68 + i
    seed = 1067 + i
    key = f"news_{book_id}"

    book_configs[key] = {
        'book_id': book_id,
        'universe': 'news',
        'seed': seed
    }

# ============================================
# Sanity check
# ============================================

missing = [k for k in book_configs if k not in benchmarks]

if missing:
    print(f"WARNING: missing from benchmarks: {missing}")
    print(f"Available keys: {sorted(benchmarks.keys())}")
else:
    print(f"All {len(book_configs)} books found in benchmarks dict")

# ============================================
# Output folder
# ============================================

output_root = (
    ""
    ""
)

# ============================================
# Choose ONE held-out seed per universe
# ============================================

folds = [
    {
        'held_out': 'default_12',
        'name': 'fold1_default'
    },
    {
        'held_out': 'scifi_49',
        'name': 'fold2_scifi'
    },
    {
        'held_out': 'news_84',
        'name': 'fold3_news'
    },
]

# ============================================
# Dataset builder
# ============================================

def build_dataset(selected_keys, held_out_key, setting_name, fold_name):

    all_jsonl = []

    held_out_seed = book_configs[held_out_key]['seed']

    print(f"\n===== {setting_name} | {fold_name} =====")
    print(f"Held-out: {held_out_key} (seed={held_out_seed})\n")

    for key in selected_keys:

        benchmark = benchmarks[key]

        if key == held_out_key:
            # SEQ ONLY
            jsonl = benchmark.build_fine_tuning_jsonl(
                answer_in_one_chapter_only=True
            )

            print(f"{key} (SEQ ONLY) -> {len(jsonl)}")

        else:
            # SEQ + MEQ
            jsonl = benchmark.build_fine_tuning_jsonl(
                answer_in_one_chapter_only=False
            )

            print(f"{key} (SEQ+MEQ) -> {len(jsonl)}")

        all_jsonl.extend(jsonl)

    shuffle(all_jsonl)

    output_dir = f"{output_root}/{setting_name}"
    os.makedirs(output_dir, exist_ok=True)

    filepath = (
        f"{output_dir}/"
        f"{setting_name}_{fold_name}_heldout_{held_out_key}"
        f"_seed_{held_out_seed}.jsonl"
    )

    with open(filepath, "w") as f:
        for item in all_jsonl:
            f.write(json.dumps(item) + "\n")

    print(f"\nTotal: {len(all_jsonl)} -> {filepath}\n")

# ============================================
# BUILD TRAINING SETS 100
# ============================================

all_books = list(book_configs.keys())

for fold in folds:

    held_out = fold['held_out']
    fold_name = fold['name']

    build_dataset(
        all_books,
        held_out,
        "training_sets_100",
        fold_name
    )